### 0. Clean repo


In [ ]:
import os

DIRECTORIES = [
    "../models", 
    # "../data/raw/files",
	"../plots",
    "../tmp",
    "../results",
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


### 1. Import & load libraries

In [ ]:
%pip cache purge
%pip install -r ../requirements.txt


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from mne import pick_types
from mne.channels import make_standard_montage
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci
from mne.preprocessing import ICA 
from mne import Epochs
from mne.decoding import CSP

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# mne.set_log_level('WARNING')
# mne.set_log_level("CRITICAL")
# mne.set_config('MNE_BROWSE_RAW_SIZE','20,8')


In [ ]:
import gc  # Garbage Collector
gc.enable()
gc.get_stats()


### 2. Info

#### Experimental Protocol

[Link of dataset](https://physionet.org/content/eegmmidb/1.0.0/)

This data set consists of over 1500 one- and two-minute EEG recordings, obtained from ***109 volunteers***, as described below.

Subjects performed different motor/imagery tasks while 64-channel EEG were recorded using the BCI2000 system (http://www.bci2000.org). Each subject performed ***14 experimental runs***: two one-minute baseline runs (one with eyes open, one with eyes closed), and three two-minute runs of each of the four following tasks:

- **TASK 1**: A target appears on either the left or the right side of the screen. The subject opens and closes the corresponding fist until the target disappears. Then the subject relaxes.
- **TASK 2**: A target appears on either the left or the right side of the screen. The subject imagines opening and closing the corresponding fist until the target disappears. Then the subject relaxes.
- **TASK 3**: A target appears on either the top or the bottom of the screen. The subject opens and closes either both fists (if the target is on top) or both feet (if the target is on the bottom) until the target disappears. Then the subject relaxes.
- **TASK 4**: A target appears on either the top or the bottom of the screen. The subject imagines opening and closing either both fists (if the target is on top) or both feet (if the target is on the bottom) until the target disappears. Then the subject relaxes.

#### Description of data:

The experimental runs were:

- Baseline, eyes open
- Baseline, eyes closed
- Task 1 (open and close left or right fist)
- Task 2 (imagine opening and closing left or right fist)
- Task 3 (open and close both fists or both feet)
- Task 4 (imagine opening and closing both fists or both feet)

Each annotation includes one of three codes (T0, T1, or T2):

- **T0** corresponds to rest
- **T1** corresponds to onset of motion (real or imagined) of
        the left fist (in runs 3, 4, 7, 8, 11, and 12)
        both fists (in runs 5, 6, 9, 10, 13, and 14)
- **T2** corresponds to onset of motion (real or imagined) of
        the right fist (in runs 3, 4, 7, 8, 11, and 12)
        both feet (in runs 5, 6, 9, 10, 13, and 14)

| Run       | Task                                |
|-----------|-------------------------------------|
| 1         | Baseline, eyes open                 |
| 2         | Baseline, eyes closed               |
| 3, 7, 11  | Motor execution: left vs right hand |
| 4, 8, 12  | Motor imagery: left vs right hand   |
| 5, 9, 13  | Motor execution: hands vs feet      |
| 6, 10, 14 | Motor imagery: hands vs feet        |


The EEGs were recorded from 64 electrodes as per the international system (excluding electrodes Nz, F9, F10, FT9, FT10, A1, A2, TP9, TP10, P9, and P10)

<img width=100% src=../images/EGG_64.png>

<img width=100% src=../images/64_channel_sharbrough.png>

### 3. Load data


In [ ]:
subject = [1]  # List of subject numbers 
run_execution = [5, 9, 13]  # Runs for executing motor tasks
run_imagery = [6, 10, 14]  # Runs for imagining motor tasks

raw_files = []

# Loop through each subject and associated runs for execution and imagery
for person_number in subject:
    for i, j in zip(run_execution, run_imagery):
        # Load EEG data for executing motor tasks
        raw_files_execution = [read_raw_edf(f, preload=True, stim_channel='auto') for f in eegbci.load_data(person_number, i)]
        raw_execution = concatenate_raws(raw_files_execution)

        # Load EEG data for imagining motor tasks
        raw_files_imagery = [read_raw_edf(f, preload=True, stim_channel='auto') for f in eegbci.load_data(person_number, j)]
        raw_imagery = concatenate_raws(raw_files_imagery)

        # Extract events and create annotations for executing motor tasks
        events, _ = mne.events_from_annotations(raw_execution, event_id=dict(T0=1, T1=2, T2=3))
        mapping = {1: 'rest', 2: 'do/feet', 3: 'do/hands'}
        annot_from_events = mne.annotations_from_events(
            events=events, event_desc=mapping, sfreq=raw_execution.info['sfreq'],
            orig_time=raw_execution.info['meas_date'])
        raw_execution.set_annotations(annot_from_events)

        # Extract events and create annotations for imagining motor tasks
        events, _ = mne.events_from_annotations(raw_imagery, event_id=dict(T0=1, T1=2, T2=3))
        mapping = {1: 'rest', 2: 'imagine/feet', 3: 'imagine/hands'}
        annot_from_events = mne.annotations_from_events(
            events=events, event_desc=mapping, sfreq=raw_imagery.info['sfreq'],
            orig_time=raw_imagery.info['meas_date'])
        raw_imagery.set_annotations(annot_from_events)

        # Append the processed raw data to the list
        raw_files.append(raw_execution)
        raw_files.append(raw_imagery)
print("✅ Done")


##### Events represent specific moments in an EEG or MEG signal, usually marked by stimuli or annotations.

In [ ]:
raw = concatenate_raws(raw_files)
originalRaw = raw.copy()

# Extract events and create annotations
events, event_dict = mne.events_from_annotations(raw)
data = raw.get_data()

# display(raw.info)
print(len(events), event_dict)
# display(raw.ch_names)
# display(data)
raw.plot(scalings=dict(eeg=1e-4));


##### Picks refer to the selection of specific channels within an EEG/MEG dataset.

In [ ]:
# filter any bad channels that were identified to have artifacts
picks = pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
print(picks)


In [ ]:
print(raw)
print(raw.info)
print(raw.annotations)
print(raw.annotations.description)
print(raw.annotations.onset)
print(raw.info['ch_names'])


In [ ]:
fig = make_standard_montage('biosemi64').plot()


In [ ]:
eegbci.standardize(raw)  # Standardize channel names
print(raw.info['ch_names'])


In [ ]:
montage = make_standard_montage('standard_1005')
raw.set_montage(montage)


### 4. Signal filtering

A Notch Filter (also called a Band-stop filter) is used to remove specific unwanted frequencies from a signal, particularly when a narrow frequency band causes interference or noise. In signal processing, the "notch" refers to the removal of frequencies within a small range, leaving the other frequencies untouched.

**How It Works:**

- Passband and Stopband:

	- The passband is the range of frequencies that the filter allows to pass through without attenuation.
	- The stopband is the range of frequencies that the filter suppresses.
	- A notch filter specifically targets and reduces a narrow band of frequencies (the notch), while passing frequencies outside of that band.

- Frequency Range:

	- The notch filter is designed to attenuate a specific frequency (or a small range of frequencies) while allowing the other frequencies to pass through. This is particularly useful when dealing with known interference frequencies, such as the power line frequency (50 Hz or 60 Hz) that can appear in many electrical signals.

**Applications of Notch Filters:** 
- Power line interference: In electrical signals, the 50 Hz or 60 Hz power line frequency is a common source of noise. The notch filter is used to remove this frequency without affecting the rest of the signal.
- Electroencephalography (EEG): In EEG data, notch filters are commonly applied to remove noise caused by electrical equipment, such as power line interference.
- Audio processing: It can also be used to remove hum or buzz noises caused by equipment, such as electrical hum from a microphone or speakers.
- Communication systems: To filter out specific unwanted frequencies or interference.

📡 EEG Frequency Bands and Their Importance

Brain activity is divided into different frequency bands:  

| **Band**  | **Frequency (Hz)** | **Function** |
|-----------|-------------------|-------------|
| **Delta** | 0.5 - 4 Hz  | Deep sleep, unconscious states |
| **Theta** | 4 - 8 Hz    | Deep relaxation, meditation |
| **Alpha** | 8 - 12 Hz   | Calm wakefulness, relaxation |
| **Beta**  | 12 - 30 Hz  | Active thinking, problem-solving |
| **Gamma** | >30 Hz      | Cognitive processing, perception |

In [ ]:
filter_data = raw.copy()

# Apply bandpass filter (between 8 and 40 Hz for motor imagery)
low_cutoff = 8
high_cutoff = 40

filter_data.filter(low_cutoff, high_cutoff, fir_design='firwin')

# filter_data.notch_filter(freqs=[60], fir_design='firwin')


In [ ]:
raw.plot_psd(fmax=80)


In [ ]:
filter_data.plot_psd(fmax=80)


In [ ]:
raw.plot(duration=20, n_channels=16, scalings=1e-4);


In [ ]:
filter_data.plot(duration=20, n_channels=16, scalings=1e-4);


In [ ]:
# Notch filter
# Remove power line noise (50Hz in Europe, 60Hz in America)
notch_freq = 60
filter_data.notch_filter(notch_freq, fir_design='firwin')


### 5. Events extraction

In [ ]:
print("events shape = ", events.shape)
print("filter_data.info['sfreq'] = ", filter_data.info['sfreq'])
print("event_id = ", event_dict)


In [ ]:
print(events)


In [ ]:
fig = mne.viz.plot_events(events, sfreq=filter_data.info['sfreq'], first_samp=filter_data.first_samp, event_id=event_dict)


### 6. ICA  Independent Component Analysis

ICA is a statistical technique used to separate mixed signals into their independent sources. It is widely used in data analysis, signal processing, and neuroscience, especially for cleaning EEG and MEG recordings.

**How does ICA work?**

Imagine you are in a room with multiple people speaking at the same time, and you record the sound with multiple microphones. Each microphone captures a mix of all voices. ICA helps to separate each individual voice from the mixed signals.

In EEG (electroencephalography) and MEG (magnetoencephalography), the brain's electrical activity is recorded using multiple sensors. However, these recordings also capture unwanted signals (artifacts) from eye movements, muscle activity, and external noise. ICA identifies these independent sources and allows us to remove unwanted components, improving data quality.

**What is ICA used for?**

EEG/MEG artifact removal: ICA helps remove eye blinks, muscle noise, and heartbeats from brain signals.

Audio signal separation: Used in speech recognition and blind source separation (e.g., separating voices in a noisy environment).

Financial and biomedical data analysis: Helps uncover hidden patterns in stock market trends or genetic data.

**Why is ICA important?**

By isolating independent sources in a dataset, ICA enhances the accuracy and reliability of data analysis, making it a crucial tool in neuroscience, machine learning, and many other fields.

In [ ]:
filter_corrected = filter_data.copy()
n_components = 20

ica = ICA(n_components=20, method='fastica', fit_params=None)

ica.fit(filter_corrected, picks=picks)

ica.plot_components()

eog_indicies, scores= ica.find_bads_eog(raw, ch_name='Fpz', threshold=1.5)
print("eog_indicies: ", eog_indicies)
ica.plot_scores(scores, exclude=eog_indicies)
ica.exclude.extend(eog_indicies)


In [ ]:
ica.plot_sources(filter_corrected);


In [ ]:
print("Number of channels:", len(filter_corrected.ch_names))
print("Total duration:", filter_corrected.times[-1], "seconds")

raw.plot(n_channels=30, start=0, duration=100, scalings=dict(eeg=1e-4));

filter_corrected.plot(n_channels=30, start=0, duration=100, scalings=dict(eeg=1e-4));


### 7. Epochs

**What are epochs in data analysis?**

In EEG analysis, epochs are short segments of data extracted from continuous recordings, centered around specific events of interest. They allow researchers to analyze brain activity before, during, and after an event, helping to study how the brain responds to stimuli.

**How do epochs work?**

Imagine recording brain activity while showing a person images. Instead of analyzing the entire recording, we extract time windows (epochs) around each image presentation, for example:

tmin = -1s → 1 second before the event

tmax = 4s → 4 seconds after the event

Each epoch corresponds to a trial where the event occurred, making it easier to analyze brain responses across multiple trials and identify consistent patterns.

**Why are epochs important?**

Improve signal clarity by focusing on specific time intervals.

Enable event-related analysis to compare brain responses to different stimuli.

Help detect artifacts by analyzing repeated patterns across trials.

Epochs are a fundamental step in cognitive neuroscience, brain-computer interfaces, and clinical diagnostics, making EEG/MEG data more structured and interpretable.

In [ ]:
tmin = -1  # Time before event in seconds
tmax = 4  # Time after event in seconds
epochs = mne.Epochs(filter_corrected, events, event_dict, tmin, tmax, proj=True, picks=picks, baseline=None, preload=True)
epochs


In [ ]:
print(f"Tmin ='{tmin}' | Tmax='{tmax}'")
times = np.linspace(tmin, tmax, 6)
print(f"Times = {times}")


In [ ]:
epochs.plot(n_channels=5, scalings=dict(eeg=1e-4));


In [ ]:
do_epoches = epochs['do']
imagine_epoches = epochs['imagine']

print(do_epoches)
print(imagine_epoches)


In [ ]:
do_hands_epoches = do_epoches['hands'].average()
do_hands_epoches.plot(spatial_colors=True, gfp=True, time_unit='s');

do_feet_epoches = do_epoches['feet'].average()
do_feet_epoches.plot(spatial_colors=True, gfp=True, time_unit='s');


In [ ]:
do_epoches['hands'].average().plot_topomap(ch_type='eeg', times=times, colorbar=True);

imagine_epoches['hands'].average().plot_topomap(ch_type='eeg', times=times, colorbar=True);


In [ ]:
do_epoches['hands'].average().plot_joint(times=times);
imagine_epoches['hands'].average().plot_joint(times=times);


In [ ]:
channelIdx = [1]

do_epoches.copy().crop(0, 4).plot_image(picks='eeg', combine='mean');

imagine_epoches.copy().crop(0, 4.0).plot_image(picks='eeg', combine='mean');


In [ ]:
print("Epoches shape          : ", epochs.get_data(copy=False).shape)
print("Epoches[do] shape      : ", do_epoches.get_data(copy=False).shape)
print("Epoches[imagine] shape : ", imagine_epoches.get_data(copy=False).shape)


In [ ]:
do_evoked = do_epoches.average()
imagine_evoked = imagine_epoches.average()


In [ ]:
diff = mne.combine_evoked([do_evoked, -imagine_evoked], weights="equal")


In [ ]:
diff.plot_joint(times=times);


In [ ]:
diff.plot_image();


### 8. CSP  Common Spatial Patterns 

CSP is a technique used in brain-computer interfaces (BCIs) and EEG signal processing to extract features that differentiate between two classes of brain activity. It is especially useful in motor imagery tasks, where a person imagines moving their left or right hand, and we want to classify these mental states.

**How CSP Works**

CSP finds spatial filters that maximize the variance of one class while minimizing the variance of the other.

These filters emphasize the most relevant brain regions for classification.

The transformed signals are then used as input for machine learning models.

**Why Is CSP Important?**

It enhances signal-to-noise ratio for EEG-based classification.

It is widely used in BCI applications, such as controlling prosthetics or communication devices for paralyzed patients.

In [ ]:
labels = {
    0: 'do/feet',
    1: 'do/hands', 
    2: 'imagine/feet',
    3: 'imagine/hands',
    4: 'rest'
}


In [ ]:
epochs_selected = epochs[['hands', 'feet']]
X = epochs_selected.get_data(copy=False )
y = epochs_selected.events[:, -1] - 1
print(y)
sfreq = epochs.info['sfreq']
print("Sample frequency: ", sfreq)


In [ ]:
csp = CSP()


In [ ]:
X_csp = csp.fit_transform(X, y)


In [ ]:
# Create a 3D array
original_3d_array = np.array([[[1, 2, 3], [4, 5, 6]],
                              [[7, 8, 9], [10, 11, 12]],
                              [[13, 14, 15], [16, 17, 18]]])

# Print the original 3D array
print("Original 3D Array:")
print(original_3d_array)
print("Shape of the Original 3D Array:", original_3d_array.shape)

# Reshape the 3D array to a 2D array
reshaped_2d_array = original_3d_array.reshape((-1, original_3d_array.shape[-1]))

# Print the reshaped 2D array
print("\nReshaped 2D Array:")
print(reshaped_2d_array)
print("Shape of the Reshaped 2D Array:", reshaped_2d_array.shape)


In [ ]:
# Assuming X is your EEG data, and X_csp is the CSP-transformed data
channel_index = 13  # Choose the channel you want to plot

# Original EEG Data
X_channel = X[:, channel_index, :]
X_2D = X_channel.reshape((X_channel.shape[0], -1))

# Scatter plot for Original EEG Data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X_2D[:, 0], X_2D[:, 1], c=y, cmap='viridis')
plt.title('Original EEG Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

# CSP-Transformed Data
X_csp_2D = X_csp.reshape((X_csp.shape[0], -1))

# Scatter plot for CSP-Transformed Data
plt.subplot(1, 2, 2)
plt.scatter(X_csp_2D[:, 0], X_csp_2D[:, 1], c=y, cmap='viridis')
plt.title('CSP-Transformed EEG Data')
plt.xlabel('Feature 1 (CSP)')
plt.ylabel('Feature 2 (CSP)')

plt.tight_layout()
plt.show()


### 9. PCA  Principal Component Analysis

**What is PCA?**

PCA is a widely used dimensionality reduction technique that helps simplify complex datasets while preserving as much important information as possible. It transforms the original features into a new set of uncorrelated variables called principal components, which capture the most variance in the data.

**How does PCA work?**
Find the Directions of Maximum Variance

PCA identifies the directions (axes) along which the data varies the most.

These new axes are called principal components.

Project Data Onto Fewer Dimensions

The data is rotated and projected onto the top principal components.

This reduces the number of dimensions while retaining the most important patterns.

**Why use PCA?**
- Reduces dimensionality, making machine learning models faster and more efficient.
- Removes noise and redundancy in datasets with correlated variables.
- Improves visualization of high-dimensional data.

In [ ]:
def analyze_eeg_pca(data, variance_threshold=96.0):
    # Store column names before transformation
    column_names = data.columns if isinstance(data, pd.DataFrame) else None

    # Data normalization
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Apply PCA
    pca = PCA()
    pca.fit(data_scaled)
    
    # Cumulative variance
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_) * 100
    n_components = np.argmax(cumulative_variance >= variance_threshold) + 1

    # Dimensionality reduction with the optimal number of components
    pca_opt = PCA(n_components=n_components)
    transformed_data = pca_opt.fit_transform(data_scaled)

    # Create DataFrame with principal components
    components_df = pd.DataFrame(
        pca_opt.components_, 
        columns=column_names,  # Restore original column names
        index=[f"PC{i+1}" for i in range(n_components)]
    )

    # Plot explained variance
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-')
    plt.axhline(y=variance_threshold, color='r', linestyle='--', label=f"{variance_threshold}% Variance")
    plt.xlabel("Number of Principal Components")
    plt.ylabel("Cumulative Explained Variance (%)")
    plt.title("PCA - Explained Variance")
    plt.legend()
    plt.grid()
    plt.show()

    return {
        "n_components": n_components,
        "components_df": components_df,
        "explained_variance": pca_opt.explained_variance_ratio_
    }

pca_data = filter_data.copy().get_data()
pca_data = pd.DataFrame(pca_data)  # Ensure the data is a DataFrame

results = analyze_eeg_pca(pca_data, variance_threshold=96.0)

print("\nPCA Analysis Summary:")
print(f"Optimal number of components: {results['n_components']}")
print("\nComponent details:")
print(results['components_df'].head(results['n_components']))
